# Data Preparation & Normalisation

**Dataset**: AG News (Text Classification)  
**Task**: Classify news articles into 4 categories: World, Sports, Business, Sci/Tech  
**Modality**: Text

In [3]:
import json
import os
import re
import string

import pandas as pd
import numpy as np

## 1. Inspect Raw Data
Examine the dataset size, structure, class distribution, and any quality issues.

In [4]:
# Load raw data
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nData types:\n{train_df.dtypes}")

Train set shape: (120000, 3)
Test set shape:  (7600, 3)

Columns: ['Class Index', 'Title', 'Description']

Data types:
Class Index     int64
Title          object
Description    object
dtype: object


In [5]:
# Preview the data
train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [6]:
# Class distribution
print("=== Train Set Class Distribution ===")
print(train_df["Class Index"].value_counts().sort_index())
print(f"\nBalance ratio (min/max): {train_df['Class Index'].value_counts().min() / train_df['Class Index'].value_counts().max():.4f}")

print("\n=== Test Set Class Distribution ===")
print(test_df["Class Index"].value_counts().sort_index())
print(f"\nBalance ratio (min/max): {test_df['Class Index'].value_counts().min() / test_df['Class Index'].value_counts().max():.4f}")

=== Train Set Class Distribution ===
Class Index
1    30000
2    30000
3    30000
4    30000
Name: count, dtype: int64

Balance ratio (min/max): 1.0000

=== Test Set Class Distribution ===
Class Index
1    1900
2    1900
3    1900
4    1900
Name: count, dtype: int64

Balance ratio (min/max): 1.0000


In [7]:
# Quality issues check
print("=== Missing Values (Train) ===")
print(train_df.isnull().sum())

print(f"\n=== Duplicate Rows ===")
print(f"Train duplicates: {train_df.duplicated().sum()}")
print(f"Test duplicates:  {test_df.duplicated().sum()}")

print(f"\n=== Text Length Statistics (Description) ===")
desc_len = train_df["Description"].str.len()
print(f"Min: {desc_len.min()}, Max: {desc_len.max()}, Mean: {desc_len.mean():.1f}, Median: {desc_len.median():.1f}")

print(f"\n=== Empty Fields ===")
print(f"Empty descriptions (train): {train_df['Description'].isna().sum() + (train_df['Description'] == '').sum()}")
print(f"Empty titles (train): {train_df['Title'].isna().sum() + (train_df['Title'] == '').sum()}")

=== Missing Values (Train) ===
Class Index    0
Title          0
Description    0
dtype: int64

=== Duplicate Rows ===
Train duplicates: 0
Test duplicates:  0

=== Text Length Statistics (Description) ===
Min: 6, Max: 985, Mean: 193.4, Median: 188.0

=== Empty Fields ===
Empty descriptions (train): 0
Empty titles (train): 0


## 2. Clean & Normalise Text Data

Text cleaning steps applied:
1. Lowercase all text
2. Remove HTML entities (`&lt;`, `#36;`, etc.)
3. Remove URLs
4. Strip punctuation
5. Remove duplicate rows
6. Drop rows with missing/empty text
7. Normalize whitespace
8. Combine title + description into a single `text` field

In [8]:
def clean_text(text):
    """Clean and normalise a text string for NLP tasks."""
    if pd.isna(text) or text is None:
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove HTML entities and tags
    text = re.sub(r'&lt;.*?&gt;', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&#?\w+;', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Remove backslashes
    text = text.replace('\\', ' ')

    # Replace #36; encoding with $
    text = re.sub(r'#36;', '$', text)

    # Strip punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
def prepare_dataset(df, split_name="train"):
    """Full cleaning pipeline for the text dataset."""
    print(f"--- Cleaning {split_name} data ---")
    original_len = len(df)

    # Remove duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  After removing duplicates: {len(df)} rows (removed {original_len - len(df)})")

    # Handle missing values
    df = df.dropna(subset=["Title", "Description"]).reset_index(drop=True)
    df = df[df["Description"].str.strip() != ""].reset_index(drop=True)
    df = df[df["Title"].str.strip() != ""].reset_index(drop=True)
    print(f"  After removing missing/empty text: {len(df)} rows")

    # Clean text columns
    df["Title_clean"] = df["Title"].apply(clean_text)
    df["Description_clean"] = df["Description"].apply(clean_text)

    # Combine title and description
    df["text"] = df["Title_clean"] + " " + df["Description_clean"]
    df["text"] = df["text"].str.strip()

    # Remove rows empty after cleaning
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    print(f"  After text cleaning: {len(df)} rows")

    # Encode labels (0-indexed)
    df["label"] = df["Class Index"] - 1

    print(f"  Final dataset: {len(df)} rows")
    return df

In [10]:
# Apply cleaning pipeline
train_clean = prepare_dataset(train_df, "train")
test_clean = prepare_dataset(test_df, "test")

--- Cleaning train data ---
  After removing duplicates: 120000 rows (removed 0)
  After removing missing/empty text: 120000 rows
  After text cleaning: 120000 rows
  Final dataset: 120000 rows
--- Cleaning test data ---
  After removing duplicates: 7600 rows (removed 0)
  After removing missing/empty text: 7600 rows
  After text cleaning: 7600 rows
  Final dataset: 7600 rows


In [11]:
# Show sample of cleaned text
print("Sample cleaned texts:")
for i in range(3):
    print(f"\n[Label {train_clean.iloc[i]['label']}] {train_clean.iloc[i]['text'][:150]}...")

Sample cleaned texts:

[Label 2] wall st bears claw back into the black reuters reuters shortsellers wall streets dwindling band of ultracynics are seeing green again...

[Label 2] carlyle looks toward commercial aerospace reuters reuters private investment firm carlyle group which has a reputation for making welltimed and occasi...

[Label 2] oil and economy cloud stocks outlook reuters reuters soaring crude prices plus worries about the economy and the outlook for earnings are expected to ...


## 3. Encode Labels & Save Mapping

Create a `id2label.json` mapping file that maps integer label IDs to human-readable class names.

In [12]:
# AG News label mapping (0-indexed)
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}

# Save mapping file
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

print("Saved id2label.json:")
print(json.dumps(id2label, indent=2))

Saved id2label.json:
{
  "0": "World",
  "1": "Sports",
  "2": "Business",
  "3": "Sci/Tech"
}


## 4. Save Prepared Dataset to Hugging Face

Push the cleaned dataset to Hugging Face Hub so it can be loaded directly in the Kaggle training notebook.

In [ ]:
from datasets import Dataset, DatasetDict

# Your Hugging Face username
HF_USERNAME = "Recurrent"
HF_DATASET_REPO = f"{HF_USERNAME}/ag-news-prepared"

# Prepare final dataframes
train_final = train_clean[["text", "label"]].copy()
test_final = test_clean[["text", "label"]].copy()

# Sample 120k from train while preserving class balance
TRAIN_SAMPLE_SIZE = 120000

train_final = (
    train_final
    .groupby("label", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=TRAIN_SAMPLE_SIZE // train_final["label"].nunique(),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_final, preserve_index=False),
    "test": Dataset.from_pandas(test_final, preserve_index=False),
})

print(dataset_dict)

# Push publicly to Hugging Face Hub
dataset_dict.push_to_hub(HF_DATASET_REPO)

print(
    f"Dataset pushed successfully:\n"
    f"https://huggingface.co/datasets/{HF_DATASET_REPO}"
)

In [ ]:
# Final summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Task: Text Classification (AG News)")
print(f"Classes: {len(id2label)} — {list(id2label.values())}")
print(f"Train samples: {len(train_final)}")
print(f"Test samples:  {len(test_final)}")
print(f"\nLabel distribution (train):")
for label_id, label_name in id2label.items():
    count = (train_final["label"] == int(label_id)).sum()
    print(f"  {label_id} ({label_name}): {count}")
print(f"\nDataset on HF Hub: https://huggingface.co/datasets/{HF_DATASET_REPO}")
print(f"id2label.json: saved locally (commit to git)")

SUMMARY
Task: Text Classification (AG News)
Classes: 4 — ['World', 'Sports', 'Business', 'Sci/Tech']
Train samples: 120000
Test samples:  7600

Label distribution (train):
  0 (World): 30000
  1 (Sports): 30000
  2 (Business): 30000
  3 (Sci/Tech): 30000

Files produced:
  - id2label.json (commit this)
  - prepared_data/train_prepared.csv (do NOT commit)
  - prepared_data/test_prepared.csv (do NOT commit)
